# Bitcoin Price Close Prediction

Uses daily close data from the last 30 days (up to yesterday) to predict today's close, then compares weighted vs flat model averaging.

In [ ]:
import csv
import os
from datetime import datetime, timedelta

import llb
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH = "btc_price.csv"

# ── Load CSV (semicolon-delimited, BOM-safe) ──────────────────────────────────
rows = []
with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter=";")
    for r in reader:
        dt = datetime.fromisoformat(r["timeOpen"].replace("Z", "+00:00")).replace(tzinfo=None)
        close = float(r["close"])
        rows.append((dt, close))

rows.sort(key=lambda x: x[0])

# ── Auto-detect today as the last date in the CSV ─────────────────────────────
today_date = rows[-1][0]
today_close = rows[-1][1]
yesterday_date = today_date - timedelta(days=1)
one_month_ago = today_date - timedelta(days=30)

# ── Build training window: one_month_ago <= date <= yesterday ─────────────────
train_rows = [(dt, c) for dt, c in rows
              if one_month_ago.date() <= dt.date() <= yesterday_date.date()]
if len(train_rows) < 2:
    raise ValueError("Not enough training data in range.")

btc_closes = [c for _, c in train_rows]
num_days = len(btc_closes)

# ── Run inference ─────────────────────────────────────────────────────────────
text = (
    "I have daily Bitcoin close prices for the past month up to yesterday. "
    "Use only these close prices to model latent level and latent volatility, "
    "then predict today's close (one-day-ahead forecast from yesterday)."
)
data = {"num_days": num_days, "btc_closes": btc_closes}
targets = ["latent_level", "latent_volatility", "future_btc_close"]

posterior = llb.infer(
        text=text,
        data=data,
        targets=targets,
        api_url="https://api.openai.com/v1/chat/completions",
        api_key="",
        api_model="gpt-4.1-mini",
        n_models=10,
        mcmc_num_warmup=200,
        mcmc_num_samples=500,
        random_seed=42,
)

# ── Weighted mean from returned posterior ─────────────────────────────────────
fut_w = np.asarray(posterior["future_btc_close"], dtype=float)
pred_w_samples = fut_w if fut_w.ndim == 1 else fut_w[:, 0]
pred_w_mean = float(np.mean(pred_w_samples))

# ── Compare prediction to today's actual close ────────────────────────────────
err = pred_w_mean - today_close
abs_err = abs(err)
pct_err = (abs_err / today_close) * 100.0

print(f"Train range       : {train_rows[0][0].date()} → {train_rows[-1][0].date()} ({num_days} days)")
print(f"Predict day       : {today_date.date()}")
print(f"Actual close      : {today_close:,.2f}")
print(f"Weighted prediction: {pred_w_mean:,.2f}")
print(f"Error             : {err:+,.2f}  (abs={abs_err:,.2f}, pct={pct_err:.2f}%)")


Number of requested models: 10
Number of generated models: 10
Number of deduplicated models: 0
Number of invalid models (syntax/parsing): 0
Number of generation request failures (timeout/network/API): 0
Number of models missing required targets: 0
Number of models that failed to compile: 0
Number of models that failed during inference: 1
Number of models dropped due to target shape mismatch: 5
Number of models dropped due to non-finite log bound: 0
Number of valid models used in final aggregation: 4
--- Model Averaging Summary ---
Target: latent_level
Number of models: 4
flat mean prediction: 68366.506117
weighted mean prediction: 68453.717688
Difference (weighted - flat): 87.211570

Top 5 models by weight for target 'latent_level':
model=1, weight=1.000000, mu_i=68453.717688
model=0, weight=0.000000, mu_i=68216.562203
model=2, weight=0.000000, mu_i=68287.012398
model=3, weight=0.000000, mu_i=68508.732180
2 least-weighted models for target 'latent_level':
model=0, weight=0.000000, mu_i

In [3]:
import csv
import os
from datetime import datetime, timedelta

import llb
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH = "btc_price.csv"

# ── Load CSV (semicolon-delimited, BOM-safe) ──────────────────────────────────
rows = []
with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter=";")
    for r in reader:
        dt = datetime.fromisoformat(r["timeOpen"].replace("Z", "+00:00")).replace(tzinfo=None)
        close = float(r["close"])
        rows.append((dt, close))

rows.sort(key=lambda x: x[0])

# ── Auto-detect today as the last date in the CSV ─────────────────────────────
today_date = rows[-1][0]
today_close = rows[-1][1]
yesterday_date = today_date - timedelta(days=1)
one_month_ago = today_date - timedelta(days=30)

# ── Build training window: one_month_ago <= date <= yesterday ─────────────────
train_rows = [(dt, c) for dt, c in rows
              if one_month_ago.date() <= dt.date() <= yesterday_date.date()]
if len(train_rows) < 2:
    raise ValueError("Not enough training data in range.")

btc_closes = [c for _, c in train_rows]
num_days = len(btc_closes)

# ── Run inference ─────────────────────────────────────────────────────────────
text = (
    "I have daily Bitcoin close prices for the past month up to yesterday. "
    "Use only these close prices to model latent level and latent volatility, "
    "then predict today's close (one-day-ahead forecast from yesterday)."
)
data = {"num_days": num_days, "btc_closes": btc_closes}
targets = ["latent_level", "latent_volatility", "future_btc_close"]

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3:latest"

posterior = llb.infer(
    text,
    data,
    targets,
    api_url=OLLAMA_URL,
    api_key=None,
    api_model=OLLAMA_MODEL,
    n_models=50,
    mcmc_num_warmup=50,
    mcmc_num_samples=100,
)

# ── Weighted mean from returned posterior ─────────────────────────────────────
fut_w = np.asarray(posterior["future_btc_close"], dtype=float)
pred_w_samples = fut_w if fut_w.ndim == 1 else fut_w[:, 0]
pred_w_mean = float(np.mean(pred_w_samples))

# ── Compare prediction to today's actual close ────────────────────────────────
err = pred_w_mean - today_close
abs_err = abs(err)
pct_err = (abs_err / today_close) * 100.0

print(f"Train range       : {train_rows[0][0].date()} → {train_rows[-1][0].date()} ({num_days} days)")
print(f"Predict day       : {today_date.date()}")
print(f"Actual close      : {today_close:,.2f}")
print(f"Weighted prediction: {pred_w_mean:,.2f}")
print(f"Error             : {err:+,.2f}  (abs={abs_err:,.2f}, pct={pct_err:.2f}%)")


Number of requested models: 50
Number of generated models: 50
Number of deduplicated models: 0
Number of invalid models (syntax/parsing): 0
Number of generation request failures (timeout/network/API): 0
Number of models missing required targets: 0
Number of models that failed to compile: 1
Number of models that failed during inference: 30
Number of models dropped due to target shape mismatch: 5
Number of models dropped due to non-finite log bound: 0
Number of valid models used in final aggregation: 14
--- Model Averaging Summary ---
Target: latent_level
Number of models: 14
flat mean prediction: 3899.943362
weighted mean prediction: 2.458493
Difference (weighted - flat): -3897.484869

Top 5 models by weight for target 'latent_level':
model=2, weight=0.742486, mu_i=0.002033
model=0, weight=0.257514, mu_i=9.541171
model=12, weight=0.000000, mu_i=22194.719744
model=8, weight=0.000000, mu_i=0.021758
model=11, weight=0.000000, mu_i=28032.927673
2 least-weighted models for target 'latent_lev